In [14]:
import pandas as pd

df = pd.read_csv('course_lead_scoring_2026.csv')

In [15]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [16]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [17]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [18]:
df = df.fillna({'lead_source': 'NA', 'industry': 'NA', 'employment_status': 'NA', 'location': 'NA', 'annual_income': 0.0, 'lead_score': 0.0})

In [19]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [20]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [21]:
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

df[numerical].corr()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.161300,0.721609,0.757204
annual_income,0.161300,1.000000,0.122842,0.229496
interaction_count,0.721609,0.122842,1.000000,0.915746
lead_score,0.757204,0.229496,0.915746,1.000000


In [22]:
corr = df[numerical].corr()

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(f'{a} & {b}: {corr.loc[a, b]:.4f}')

interaction_count & lead_score: 0.9157
number_of_courses_viewed & lead_score: 0.7572
number_of_courses_viewed & interaction_count: 0.7216
annual_income & interaction_count: 0.1228


In [23]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [24]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [25]:
from sklearn.metrics import mutual_info_score

categorical = ['lead_source', 'industry', 'employment_status', 'location']

In [27]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [28]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

In [31]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(round(accuracy, 2))

0.65


In [32]:
def train_and_score(features):
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return accuracy_score(y_val, model.predict(X_val))


all_features = categorical + numerical
base_accuracy = train_and_score(all_features)
print('base accuracy:', base_accuracy)

for feature in all_features:
    features = [f for f in all_features if f != feature]
    acc = train_and_score(features)
    diff = base_accuracy - acc
    print(f'{feature:28s} acc={acc:.4f}  diff={diff:.4f}')

base accuracy: 0.645
lead_source                  acc=0.6420  diff=0.0030
industry                     acc=0.6450  diff=0.0000
employment_status            acc=0.6450  diff=0.0000
location                     acc=0.6450  diff=0.0000
number_of_courses_viewed     acc=0.6430  diff=0.0020
annual_income                acc=0.7240  diff=-0.0790
interaction_count            acc=0.6010  diff=0.0440
lead_score                   acc=0.6440  diff=0.0010


In [34]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)
    print(c, round(accuracy, 2))

1e-06 0.6
1e-05 0.6
0.0001 0.61
0.001 0.65
